# HPO SolarSystemLander — KISS

S1 optimizes the exploration schedule for 1,000 training episodes.

In [ ]:
# setup Colab
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys
from unittest.mock import Mock

import torch
from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from dqn.vector_training import VectorTrainingConfig
from hpo.drive_backup import backup_to_drive, restore_from_drive
from hpo.evaluation.scoring import ScoringConfig
from hpo.lunar_lander.logging import configure_file_logging
from hpo.objective import TrialConfig
from hpo.solar_system_lander.environment import EnvFactory
from hpo.study import StudyRunner

drive.mount("/content/drive")
LOCAL_DIR = Path("/content/rl_lab/hpo/runs")
DRIVE_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

DB_NAME = "solar_system_lander_8d_kiss.db"
LOG_NAME = "solar_system_lander_8d_kiss.log"
LOCAL_DB = LOCAL_DIR / DB_NAME
LOCAL_LOG = LOCAL_DIR / LOG_NAME
configure_file_logging(LOCAL_DIR, LOG_NAME)


In [ ]:
# ----------------------------------------
# define search space

PARAMS = {
    "learning_rate": 0.0022727854024196057,
    "batch_size": 512,
    "eps_end": 0.047716002108220544,
    "eps_decay": 43_214,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 2_500,
    "optimize_every": 2,
    "replay_memory_capacity": 1_200_000,
    "num_episodes": 1_000,
}

class SearchSpace1:
    def training_config(self, trial, params):
        return VectorTrainingConfig(
            num_episodes=params["num_episodes"],
            batch_size=params["batch_size"],
            gamma=params["gamma"],
            eps_start=1.0,
            eps_end=trial.suggest_float("eps_end", 0.03, 0.08),
            eps_decay=trial.suggest_int(
                "eps_decay", 43_000, 150_000, log=True
            ),
            tau=params["tau"],
            learning_rate=params["learning_rate"],
            learning_starts=params["learning_starts"],
            optimize_every=params["optimize_every"],
        )

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

In [ ]:
# ----------------------------------------
# run

environment_factory = EnvFactory("8d")
scoring_cfg = ScoringConfig(
    quality_weight=0.95,
    eval_episodes=10,
    baseline_env_steps=112_314.33333333333,
    baseline_processed_samples=28_491_093.333333332,
)
runner = StudyRunner(
    storage_path=lambda _name: LOCAL_DB,
    environment_factory=environment_factory,
    trial_cfg=TrialConfig(
        num_envs=20,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    ),
    incumbent_params=PARAMS,
    reporter=Mock(),
    study_attrs=environment_factory.metadata(),
    robust_candidates=1,
    extra_seeds=(),
    sync_fn=backup,
)

runner.run("s1_exploration_1000", SearchSpace1(), 25, scoring_cfg)